In [1]:
# ==== TRG-Diffusion v13.2: low-light RGB + thermal -> clear RGB  (v2: verbose) ====
# Kaggle: Accelerator GPU T4, Internet ON, run in an INTERACTIVE session (not "Save Version").
# Same model/inference recipe as app_v13_2.py; v2 only adds visible progress, a single fast file
# search, a copy with progress, and prints the share URL explicitly.
import os, sys, shutil, subprocess, time

T0 = time.time()
def log(*a):
    print(f"[{time.time() - T0:6.0f}s]", *a, flush=True)

def sh(cmd):
    """Run a command and stream its output INTO the notebook cell (plain subprocess.run goes to the
    kernel log, which is why the old cell looked silent)."""
    log("$", cmd if isinstance(cmd, str) else " ".join(cmd))
    p = subprocess.Popen(cmd, shell=isinstance(cmd, str), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print("   ", line.rstrip(), flush=True)
    if p.wait():
        raise RuntimeError(f"command failed with code {p.returncode}: {cmd}")

# ---------------------------------------------------------------- 0. dependencies
log("stage 0: pip install")
sh([sys.executable, "-m", "pip", "install", "--progress-bar", "off", "einops", "omegaconf", "kornia", "gradio"])
try:
    import pytorch_lightning  # noqa: F401
except ImportError:
    sh([sys.executable, "-m", "pip", "install", "--progress-bar", "off", "pytorch-lightning"])

# ---------------------------------------------------------------- 1. locate files (one walk of /kaggle/input)
log("stage 1: scanning /kaggle/input")
WEIGHTS_DIR = "/kaggle/input/datasets/rushikeshkulkarni401/v13-2-best-weights/required_weights"
BEST_SRC = f"{WEIGHTS_DIR}/v13.2/best.pt"            # v13.2 best.pt (step 2250)
SLIM_SRC = f"{WEIGHTS_DIR}/viis_slim.ckpt"     # VIIS base weights, EMA folded in
for p in (BEST_SRC, SLIM_SRC):
    if os.path.isfile(p):
        log(f"found {p} ({os.path.getsize(p) / 1e9:.2f} GB)")
    else:
        log(f"MISSING {p}; contents of {WEIGHTS_DIR}:",
            os.listdir(WEIGHTS_DIR) if os.path.isdir(WEIGHTS_DIR) else "(folder does not exist)")
        raise FileNotFoundError(p)
WANT = {"train_v13.py": None}                  # only the code still needs to be located
n = 0
for d, dirs, files in os.walk("/kaggle/input"):
    n += 1
    for f in files:
        p = os.path.join(d, f)
        for k in WANT:
            if WANT[k] is None and p.endswith("/" + k):
                WANT[k] = p
    if all(v is not None for v in WANT.values()):
        break
log(f"scanned {n} folders")
for k, v in WANT.items():
    log(f"  {k:20s} -> {v}")

def copy_with_progress(src, dst, chunk=64 << 20):
    size, done, last = os.path.getsize(src), 0, time.time()
    log(f"copying {src} ({size / 1e9:.2f} GB) -> {dst}")
    with open(src, "rb") as fi, open(dst + ".part", "wb") as fo:
        while True:
            b = fi.read(chunk)
            if not b:
                break
            fo.write(b); done += len(b)
            if time.time() - last > 10:
                log(f"  {done / 1e9:.2f}/{size / 1e9:.2f} GB"); last = time.time()
    os.replace(dst + ".part", dst)
    log("copy done")

# ---------------------------------------------------------------- 2. code
REPO = "/kaggle/working/VIIS"
if not os.path.exists(f"{REPO}/trg13/sampler.py"):
    assert WANT["train_v13.py"], "VIIS_patched_v13 code not found under /kaggle/input (attach the code dataset)"
    shutil.copytree(os.path.dirname(WANT["train_v13.py"]), REPO, dirs_exist_ok=True)
sys.path.insert(0, REPO); os.chdir(REPO)
log("code at", REPO)

# ---------------------------------------------------------------- 3. weights
BEST = BEST_SRC                                 # 0.6 GB, read once by torch.load: fine from /kaggle/input
SLIM = "/kaggle/working/viis_slim.ckpt"         # local copy -> fast memory-mapped load
if not os.path.exists(SLIM) or os.path.getsize(SLIM) != os.path.getsize(SLIM_SRC):
    copy_with_progress(SLIM_SRC, SLIM)

# ---------------------------------------------------------------- 4. model
log("stage 4: importing torch / trg13")
import numpy as np, cv2, torch
from trg13 import model as MD, sampler as SM, align as AL

assert torch.cuda.is_available(), "no GPU: set Accelerator = GPU T4"
log("GPU:", torch.cuda.get_device_name(0))
log("building model (VIIS + adapter) ...")
model = MD.build(REPO, SLIM, device="cuda", mode="attn_out", use_tpa=True)
meta, step, _ = MD.ckpt_meta(BEST)
log(f"loading {BEST} (step {step}, {meta}) ...")
MD.load_ckpt(model, BEST)
model.eval()
log(f"model ready, GPU mem {torch.cuda.memory_allocated() / 1e9:.1f} GB")

# ---------------------------------------------------------------- 5. inference
def to_gray_u8(img):
    img = np.asarray(img)
    if img.ndim == 3:
        img = cv2.cvtColor(np.ascontiguousarray(img[..., :3]), cv2.COLOR_RGB2GRAY)
    if img.dtype != np.uint8:
        img = np.clip(img * (255 if img.max() <= 1.0 else 1), 0, 255).astype(np.uint8)
    return img

def to_rgb_u8(img):
    img = np.asarray(img)
    if img.ndim == 2:
        img = np.repeat(img[..., None], 3, 2)
    img = img[..., :3]
    if img.dtype != np.uint8:
        img = np.clip(img * (255 if img.max() <= 1.0 else 1), 0, 255).astype(np.uint8)
    return np.ascontiguousarray(img)

@torch.no_grad()
def enhance_pair(rgb, thermal, align=True, long_side=512, steps=25, seed=123, keep_size=True):
    rgb, ir = to_rgb_u8(rgb), to_gray_u8(thermal)
    H, W = rgb.shape[:2]
    if ir.shape[:2] != (H, W):
        ir = cv2.resize(ir, (W, H), interpolation=cv2.INTER_AREA)
    info = {}
    if align:
        ir, warp, ok, cc = AL.align_pair(rgb, ir)
        info.update(aligned=bool(ok), ecc=round(float(cc), 4) if cc == cc else None)
    r, i = SM.prep_frame(rgb, ir, long_side)
    t = time.time()
    with MD.ema_scope(model):
        out01 = SM.enhance(model, r, i, steps=steps, eta=0.0, seed=seed, guid=1.0)
    info.update(seconds=round(time.time() - t, 1), model_size=f"{r.shape[1]}x{r.shape[0]}")
    out = (np.clip(out01, 0, 1) * 255 + 0.5).astype(np.uint8)
    if keep_size and out.shape[:2] != (H, W):
        out = cv2.resize(out, (W, H), interpolation=cv2.INTER_CUBIC)
    return out, info

def enhance_files(rgb_path, ir_path, out_path, **kw):
    rgb = cv2.cvtColor(cv2.imread(rgb_path, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    ir = cv2.imread(ir_path, cv2.IMREAD_GRAYSCALE)
    out, info = enhance_pair(rgb, ir, **kw)
    cv2.imwrite(out_path, cv2.cvtColor(out, cv2.COLOR_RGB2BGR))
    return info

# ---------------------------------------------------------------- 6. web interface
log("stage 6: starting Gradio")
import gradio as gr
log("gradio", gr.__version__)

def ui_fn(rgb, thermal, align, long_side, steps, seed):
    if rgb is None or thermal is None:
        raise gr.Error("Upload both the low-light RGB image and its thermal image.")
    return enhance_pair(rgb, thermal, align=align, long_side=int(long_side), steps=int(steps), seed=int(seed))

with gr.Blocks(title="Thermal-guided low-light enhancement (v13.2)") as demo:
    gr.Markdown("## Thermal-RGB guided diffusion: low-light RGB + thermal -> clear RGB\n"
                "VIIS fine-tuned (v13.2, step 2250). ~20 s per image at long side 512 on a T4.")
    with gr.Row():
        rgb_in = gr.Image(label="Low-light RGB", type="numpy")
        ir_in = gr.Image(label="Thermal (IR)", type="numpy")
        out_img = gr.Image(label="Enhanced RGB", type="numpy", format="png")
    with gr.Row():
        align_cb = gr.Checkbox(value=True, label="Auto-align thermal to RGB (ECC)")
        ls = gr.Slider(256, 1024, value=512, step=32, label="Processing long side (px)")
        st = gr.Slider(10, 50, value=25, step=5, label="DDIM steps")
        sd = gr.Number(value=123, precision=0, label="Seed")
    info_out = gr.JSON(label="Info")
    gr.Button("Enhance", variant="primary").click(ui_fn, [rgb_in, ir_in, align_cb, ls, st, sd], [out_img, info_out])

demo.queue(max_size=8).launch(share=True, prevent_thread_lock=True, inline=False, quiet=False)
log("local URL :", demo.local_url)
log("PUBLIC URL:", demo.share_url)
if not demo.share_url:
    log("No share URL: check Internet is ON for this session (the share tunnel downloads a small frpc binary).")

[     0s] stage 0: pip install
[     0s] $ /usr/bin/python3 -m pip install --progress-bar off einops omegaconf kornia gradio
      Attempting uninstall: pillow
        Found existing installation: pillow 13.0.0
        Uninstalling pillow-13.0.0:
          Successfully uninstalled pillow-13.0.0
    ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
    dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
[    33s] stage 1: scanning /kaggle/input
[    33s] found /kaggle/input/datasets/rushikeshkulkarni401/v13-2-best-weights/required_weights/v13.2/best.pt (0.61 GB)
[    33s] found /kaggle/input/datasets/rushikeshkulkarni401/v13-2-best-weights/required_weights/viis_slim.ckpt (2.46 GB)
[    34s] scanned 5 folders
[    34s]   train_v13.py         -> /kaggle/input/datasets/rushikeshkulkarni401/trg-v13-2/VIIS_patched_v13/train_v13.py
[  

/kaggle/working/VIIS/informationSynthesis/ms_deform_attn.py:83: SyntaxWarning: invalid escape sequence '\s'
  :param input_flatten               (N, \sum_{l=0}^{L-1} H_l \cdot W_l, C)
/kaggle/working/VIIS/informationSynthesis/ms_deform_attn.py:45: UserWarning: You'd better set d_model in MSDeformAttn to make the dimension of each attention head a power of 2 which is more efficient in our CUDA implementation.
  warnings.warn("You'd better set d_model in MSDeformAttn to make the dimension of each attention head a power of 2 "


making attention of type 'vanilla' with 1024 in_channels
making attention of type 'vanilla' with 1024 in_channels
DiffusionWrapper has 558.47 M params.
Keeping EMAs of 798.
making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 32, 32) = 3072 dimensions.
making attention of type 'vanilla' with 512 in_channels
Using first stage also as cond stage.
[v13] loading VIIS weights from /kaggle/working/viis_slim.ckpt (2.5 GB, mmap) ...
[v13] VIIS weights loaded in 0s: missing 0 unexpected 0 (slim: EMA already folded in)


/usr/local/lib/python3.13/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


[v13] trainable mode=attn_out: 76.7M of 616.7M (adapter 2.91M)
[   100s] loading /kaggle/input/datasets/rushikeshkulkarni401/v13-2-best-weights/required_weights/v13.2/best.pt (step 1250, {'mode': 'attn_out', 'use_tpa': True, 'ema_decay': 0.999}) ...
[   100s] model ready, GPU mem 2.8 GB
[   100s] stage 6: starting Gradio
[   103s] gradio 6.30.0
* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://57e6ed73a443e4580d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
[   105s] local URL : http://127.0.0.1:7860/
[   105s] PUBLIC URL: https://57e6ed73a443e4580d.gradio.live
